# Preprocessing (filtering, ICA, and creating epochs)

In this notebook, you will explore some of the most common preprocessing steps used when working with MEG data. The code and concepts covered here correspond to the `03_filter.py`, `04_ica.py`, and `05_epochs.py` scripts. We will also make use of some of our *bookkeeping* tools, such as the `FileNames` object, based on the approach described by [Van Vliet (2020)](https://pmc.ncbi.nlm.nih.gov/articles/PMC7098566/). 

The purpose of having a notebook alongside the scripts is to give you a space to experiment. You can try different parameter settings, inspect the data using MNE's interactive plots, and get a feel for what happens to the data at each step of the preprocessing pipeline. It is also useful to inspect which channels are bad and should be excluded from analysis!

In [ ]:
import mne
from pathlib import Path
%matplotlib widget 

# importing from our local utils
import sys
sys.path.append("../scripts_skeleton/")
from utils.fnames import FileNames

## Loading data
We will start by loading the data from one subject. You can choose the subject you collected with your study group! This is a good chance to introduce the `FileNames` class, which you'll be using during the class!

### FileNames

In [ ]:
# Initialise the filenames object
fnames = FileNames()

# define the root directory for all the data
fnames.add('root', '/work')

# based on the previously defined 'root' we can now define the raw and subjects_dir
# TODO: remove the _ before raw
fnames.add('raw', '{root}/_raw')

# predefine path to raw file of one subject with placeholders for subject name and date! 
# TODO: Change with appropriate file name
fnames.add('sub_raw', '{raw}/{subject}/{date}/workshop_2025_raw.fif')

To get the path to a given subject's raw file, we can run the following code:

In [ ]:
raw_path = fnames.sub_raw(subject="0164", date="20251003_000000")
print(raw_path)

### Using MNE to load the data

In [ ]:
raw = mne.io.read_raw_fif(raw_path, preload=True)

In [ ]:
# we will crop the data just for the purpose of playing around with the data
raw.crop(150, 600)

## Marking bad channels

### Inspecting raw time courses


In [ ]:
raw.plot();

In [ ]:
# we'll also try filtering and plotting again
filtered_raw = raw.copy()
filtered_raw.filter(0.1, 40)

filtered_raw.plot();

In [ ]:
# we will filter later. deleting for now to avoid memory issues
del filtered_raw

### Identifying bad channels from power spectral density (psd)
Besides scrolling through the raw data, it can be useful to compute the power spectral density. Often bad channels are easier to identify here than in the raw traces The X-axis contains the frequency (Hz) and the y-axis the power of each for each frequency in dB.

In [ ]:
raw.compute_psd().plot();

### Mark identified bad channels in the raw.
Provide the bad channels as a list of strings.

In [ ]:
raw.info["bads"] = ["MEG2412"]

## Filtering

### Playing with filtering parameters
With filtering, we can reduce the contributions of frequencies that contain signal that is not of interest to our analysis. Just to get an intuition about what happens when we filter the data, lets apply different filters!

In [ ]:
copy_lowpass = raw.copy()
copy_lowpass.filter(h_freq=40, l_freq=None) ## lowpass filter of 40 Hz

copy_highpass = raw.copy()
copy_highpass.filter(h_freq=None, l_freq=1) ## highpass filter of 1 Hz

copy_bandstop = raw.copy()
copy_bandstop.filter(h_freq=1, l_freq=40) ## bandstop filler of 1-40 Hz

copy_bandpass = raw.copy()
copy_bandpass.filter(h_freq=40, l_freq=1); ## bandpass fillter of 1-40 Hz

In [ ]:
# TODO: For each of the copies, compute a psd, plot it and ascertain for yourself what they do and not do!

In [ ]:
# del to avoid memory issues
del copy_lowpass, copy_highpass, copy_bandstop, copy_bandpass

### Choose a filter for the raw data
Choose a filter for the raw data, and apply it! The exact parameters depends on the planned analysis. I suggest using a bandpass filter between `0.1` and `40` for now. 

In [ ]:
raw.filter(0.1, 40)

## ICA
If you are interested, have a look at the [mne tutorial about ICA](https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html). Here you'll find much more information about how it works. 

First, we load some needed functions from MNE!

In [ ]:
from mne.preprocessing import ICA, create_ecg_epochs, create_eog_epochs


### Applying a higher highpass filter


In [ ]:
raw_ica_fit = raw.copy().filter(1, None) # lowpass already applied

### Fit ICA

In [ ]:
N_ICA_COMPONENTS = 0.99
ICA_METHOD = "fastica"
RANDOM_STATE = 42
ICA_DECIM = 10

ica = mne.preprocessing.ICA(n_components=N_ICA_COMPONENTS, random_state=RANDOM_STATE, method=ICA_METHOD)
ica.fit(raw_ica_fit, decim=ICA_DECIM)

### Visualise

In [ ]:
ica.plot_components();
ica.plot_sources(raw);

### Automatic detection of components related to heart activity and eye movement

#### Heart activity

In [ ]:
ecg_epochs = create_ecg_epochs(raw_ica_fit, tmin=-0.3, tmax=0.3, preload=False)
    

ecg_epochs.decimate(5)
ecg_epochs.load_data()
ecg_epochs.apply_baseline((None, None))
ecg_inds, ecg_scores = ica.find_bads_ecg(ecg_epochs, method="ctps")


# barplot of ICA component match scores
ica.plot_scores(ecg_scores);

# plot diagnostics
for pick in ecg_inds:
    ica.plot_properties(raw, picks=pick);

# plot ICs applied to raw data, with matches highlighted
ica.plot_sources(raw, show_scrollbars=False);


#### Eye movement

In [ ]:
eog_epochs = create_eog_epochs(raw_ica_fit, tmin=-0.5, tmax=0.5, preload=False)

# add the rest of the code yourself!

#### Apply to the data

In [ ]:
ica.exclude = ecg_inds + eog_inds

ica.apply(raw)

In [ ]:
# have a look at the data!
raw.plot();

## Epoching the data

### Finding events



In [ ]:
events = mne.find_events(raw, stim_channel="STI101", min_duration=0.002)

### Create a dictionary of what each event ID represents 
Hint: Check the set_experiment parameters methods in the `Experiment` class [here](https://github.com/laurabpaulsen/2026_advanced_cognitive_neuroscience/blob/main/week37/MEG_experiment/subjective_experience_v2.py)!

In [3]:
event_id = {
    'stimulus/0': 1,
    # insert the rest of the events here!
}

#### Exercise: Update `event_id` based on the events presented during an experiment

When creating epochs, MNE will raise an error if the `event_id` dictionary contains event descriptions for triggers that did not actually occur in the data.

Because different participants or experimental runs may contain different sets of events, we need to update `event_id` for each run so that it only contains events that were actually presented.

**Your task:**
Create a function that takes:

1. An `event_id` dictionary containing all possible event descriptions and their corresponding event codes.
2. The events that were actually presented during a given experimental run.

The function should return a **new `event_id` dictionary containing only the events that occurred in that run**.

Add the function to the `utils` directory so that it can be reused during preprocessing.

**Example goal:**

If the original `event_id` contains five possible events, but only three of them were presented in a particular run, the returned dictionary should contain only those three events.


In [2]:
def update_event_id(events, event_id):
    # insert your code here!
    pass

### Plotting events


In [ ]:
mne.viz.plot_events(events, event_id=event_ids, first_samp=raw.first_samp);

### Creating the epochs

In [ ]:
tmin, tmax = -0.2, 0.5

epochs = mne.Epochs(
    raw, 
    events = events, 
    event_id = event_id, 
    tmin = tmin, 
    tmax = tmax,  
    baseline = (tmin, None), 
    reject = None,
    preload = True
)

### Resample

In [ ]:
epochs.resample(250)